# 第4章：词嵌入（Embedding）的奥秘

> **预计学习时间：约 30 分钟**
>
> 本章将带你理解如何把文字变成数字，以及为什么这个过程如此重要。我们每天都在和文字打交道，但计算机只认数字——词嵌入就是连接"人类语言"和"机器数学"的桥梁。读完本章后，你将理解"国王 - 男人 + 女人 ≈ 女王"背后的数学原理。

**运行环境**: Kaggle Notebook (CPU 即可，词嵌入计算量较小)
**前置要求**: 完成第3章张量与神经网络基础

---
## 0. 环境准备

In [ ]:
# 安装 gensim（用于加载预训练词向量）
# 限制 smart_open < 7，避免 Python 3.12 + Kaggle 环境下的循环导入问题
# 如果仍然失败，后续代码会自动切换到手动下载方式，不影响学习
!pip install -q 'smart_open>=6.2.0,<7.0.0' gensim 2>/dev/null

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA
from collections import Counter

# 设置随机种子
torch.manual_seed(42)
np.random.seed(42)

# 设置字体
plt.rcParams['font.size'] = 12

print(f"PyTorch 版本: {torch.__version__}")
print("本章主要使用 CPU，无需 GPU")

---
## 1. 为什么需要把文字变成数字

### 生活中的"信息 → 数字"

其实我们每天都在把信息转化成数字，只是很少注意到：

> **颜色** — "红色"在计算机里是 RGB(255, 0, 0)，三个数字就能精确描述任何颜色
>
> **位置** — "北京"在地图上是经纬度 (116.4, 39.9)，两个数字就能定位全球任何地点
>
> **温度** — "很热"对每个人感受不同，但 38.5°C 是精确的、可比较的
>
> **考试成绩** — "学得不错"很模糊，但 92 分可以精确排名和比较
>
> **共同点：要让信息可以被精确处理和比较，都需要先变成数字！**

### 大模型面对的挑战

```
"今天天气真好" → ??? → 神经网络(矩阵乘法) → 输出

中间这个 ??? 就是词嵌入要解决的核心问题：
如何把文字变成数字，而且要"变得好"
```

**什么叫"变得好"？**

**不好的编码：**
```
"cat" → 1,  "dog" → 2,  "plane" → 3
问题: 2 - 1 = 1 = 3 - 2
"dog - cat" 和 "plane - dog" 的距离一样？
这种编码完全无法反映语义关系！
```

**好的编码：**
```
"cat"   → [0.8, 0.2, 0.9, ...]     # 一个向量
"dog"   → [0.7, 0.3, 0.8, ...]     # 和 cat 很接近！
"plane" → [-0.1, 0.9, -0.3, ...]   # 和 cat/dog 很远
语义相似 → 数字也相似！
```

### 类比：地图上的坐标

```
方案 A：随机编号（类似简单的整数编码）

  北京 = 1, 上海 = 2, 广州 = 3, 纽约 = 4, 伦敦 = 5

  问题：编号无法反映任何关系

方案 B：坐标（类似词嵌入）

  北京 = (116.4, 39.9)
  上海 = (121.5, 31.2)   ← 数字接近 → 地理上也接近！
  广州 = (113.3, 23.1)
  纽约 = (-74.0, 40.7)   ← 数字差距大 → 地理上也很远！
  伦敦 = (-0.1, 51.5)

  坐标天然编码了空间关系！

词嵌入做的就是同样的事：
  给每个词一个"语义坐标"
  语义相近 → 坐标接近
  语义不同 → 坐标远离
```

> 💡 **记忆要点**
> - 神经网络只能处理数字，所以必须把文字转成数字
> - 好的编码应该让**语义相近的词在数字上也相近**
> - 词嵌入 = 给每个词一个"语义坐标"，就像给城市标经纬度

---
## 2. One-hot 编码：最简单但最笨的方法

### 什么是 One-hot 编码

```
One-hot：用一个只有一个位置是 1 的向量表示每个词

词表: ["cat", "dog", "fish", "bird", "plane"]

  "cat"   → [1, 0, 0, 0, 0]
  "dog"   → [0, 1, 0, 0, 0]
  "fish"  → [0, 0, 1, 0, 0]
  "bird"  → [0, 0, 0, 1, 0]
  "plane" → [0, 0, 0, 0, 1]

类似一个布尔数组: isCat = [true, false, false, false, false]
```

### One-hot 的三大致命问题

![编码方式对比](images/encoding_comparison.png)

*图1：One-hot 编码 vs 稠密嵌入对比*

**问题1：维度爆炸**

| 模型 | 词表大小 |
|------|----------|
| 中文小模型 | ~20,000 |
| BERT | 30,522 |
| GPT-2 | 50,257 |
| GPT-4 / LLaMA | ~100,000+ |

50,000 个词 → 每个词的 One-hot 向量就是 50,000 维，其中 49,999 个是 0。极度浪费！

**问题2：无法表达语义相似性**

```
"cat" = [1, 0, 0, 0, 0]
"dog" = [0, 1, 0, 0, 0]

余弦相似度(cat, dog) = 0

所有不同词对的相似度都是 0！
cat vs dog = 0, cat vs plane = 0 ... 这完全说不通。
```

**问题3：无法泛化**

模型学到了 "cat likes eating fish"，但看到 "dog likes eating bones" 时完全帮不上忙。因为在 One-hot 下 "cat" 和 "dog" 毫无关系。

**动手验证**：让我们用代码亲眼看看 One-hot 的问题。

In [ ]:
# =============================================
# 2.1 创建 One-Hot 向量
# =============================================

# 定义一个小词表
vocab = ["cat", "dog", "fish", "bird", "king", "queen", "man", "woman"]
vocab_size = len(vocab)
word_to_idx = {word: idx for idx, word in enumerate(vocab)}

print(f"词表大小: {vocab_size}")
print(f"词到索引的映射: {word_to_idx}\n")

# 创建 One-Hot 向量
def to_one_hot(word, word_to_idx, vocab_size):
    vec = torch.zeros(vocab_size)
    vec[word_to_idx[word]] = 1.0
    return vec

# 展示每个词的 One-Hot 向量
print("One-Hot 编码:")
for word in vocab:
    vec = to_one_hot(word, word_to_idx, vocab_size)
    print(f"  {word:>6} → {vec.tolist()}")

In [ ]:
# =============================================
# 2.2 One-Hot 的问题：无法表达语义相似度
# =============================================

def cosine_similarity(v1, v2):
    dot = torch.dot(v1, v2)
    norm = torch.norm(v1) * torch.norm(v2)
    return (dot / norm).item() if norm > 0 else 0.0

# 比较几对词的相似度
pairs = [("cat", "dog"), ("cat", "fish"), ("king", "queen"), ("king", "fish")]

print("One-Hot 向量的余弦相似度:")
print("=" * 45)
for w1, w2 in pairs:
    v1 = to_one_hot(w1, word_to_idx, vocab_size)
    v2 = to_one_hot(w2, word_to_idx, vocab_size)
    sim = cosine_similarity(v1, v2)
    print(f"  {w1:>6} vs {w2:<6}  相似度 = {sim:.2f}")

print("\n问题: 所有词对的相似度都是 0！")
print("  → 'cat' 和 'dog' 明明语义相近，但 One-Hot 说它们完全不相关")

In [ ]:
# =============================================
# 2.3 可视化 One-Hot 的问题
# =============================================
# 用 PCA（主成分分析）把高维的 One-Hot 向量降到 2 维，画在平面上
# PCA 是一种常用的降维方法：把高维数据投影到低维，同时尽量保留信息

# 构造 One-Hot 矩阵：8个词 → 8x8 的单位矩阵
# 每一行就是一个词的 One-Hot 向量
one_hot_matrix = torch.eye(vocab_size).numpy()

# PCA 降维：从 8 维降到 2 维，这样才能画在平面上
pca = PCA(n_components=2)
one_hot_2d = pca.fit_transform(one_hot_matrix)

fig, ax = plt.subplots(figsize=(8, 6))

# 画散点图：每个点代表一个词
ax.scatter(one_hot_2d[:, 0], one_hot_2d[:, 1], s=100, c='steelblue', zorder=5)

# 在每个点旁边标注词的名字
# xytext=(8,8) 表示文字偏移量，避免和点重叠
for i, word in enumerate(vocab):
    ax.annotate(word, (one_hot_2d[i, 0], one_hot_2d[i, 1]),
                textcoords="offset points", xytext=(8, 8), fontsize=12)

ax.set_title('One-Hot Vectors 2D Visualization (PCA)', fontsize=14)
ax.set_xlabel('PC1')
ax.set_ylabel('PC2')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("观察: One-Hot 向量在空间中均匀分布，所有词距离相等")
print("  → 无法体现语义关系（cat/dog 应该比 cat/king 更近）")

> 💡 **记忆要点**
> - One-hot 编码：一个位置是 1，其余全是 0
> - 三大问题：**维度爆炸**、**无法表达语义相似性**、**无法泛化**
> - 词汇表有 5 万个词 → 每个词就是 5 万维的稀疏向量，99.998% 是 0
> - 任何两个不同词的相似度都是 0，这完全不合理

---
## 3. 词嵌入的突破：让相似的词靠在一起

### 核心思想：用稠密向量代替稀疏向量

```
One-hot（50,000 维，稀疏）:
  "cat" → [0, 0, ..., 0, 1, 0, ..., 0, 0]
           ← 49,999 个 0，1 个 1 →

Embedding（256 维，稠密）:
  "cat" → [0.23, -0.45, 0.78, 0.12, -0.34, ..., 0.56]
           ← 256 个有意义的数字 →
```

**类比：描述一个人**

- **One-hot 方式**：世界上有 80 亿人，给每人一个编号。张三 = 第 1,234,567,890 号 → 编号不包含任何信息
- **词嵌入方式**：张三 = [height:175, weight:70, age:30, ...] → 这些数字直接反映特征，而且可以比较

### 词向量的空间含义

![词向量空间](images/word_vector_space.png)

*图2：二维词向量空间中的语义关系*

```
词在坐标系中 —— 语义相近的词聚集在一起：

  维度 2（动物特征）
  |
  |  . cat          . dog
  |
  |      . goldfish
  |
  |                        . car
  |
  |                  . plane
  |                              . rocket
  |
  +-------------------------------------> 维度 1（交通工具特征）

真实的词向量有 256 到 4096 个维度。
每个维度可能编码一种语义特征
（动物性、大小、情感倾向等）
这些维度是从数据中自动学到的！
```

### 词嵌入的核心假设

**分布式假设（Distributional Hypothesis）：**

> "You shall know a word by the company it keeps." — J.R. Firth, 1957

```
"cat likes eating ____"
"dog likes eating ____"

"cat" 和 "dog" 出现在相似的上下文中
  → 它们含义相似
  → 它们的向量应该接近

类比 Netflix 推荐系统：
  相似用户看过的电影 → 类型相似 → 互相推荐
  相似上下文中的词 → 含义相似 → 向量接近
```

> 💡 **记忆要点**
> - 词嵌入 = 用**稠密、低维**的向量代替**稀疏、高维**的 One-hot 向量
> - 从 50,000 维降到 256 维，每个维度都携带有意义的语义信息
> - 核心假设：**出现在相似上下文中的词，含义相似，向量接近**

---
## 4. PyTorch 中的 Embedding 层

### 从 One-hot 到 Embedding 的桥梁

Embedding 层本质上是一个**查表**操作：

```
第 1 步：建一张查找表（嵌入矩阵）

  ID    词       向量
  --    ----     ------
   0    cat   → [0.23, -0.45, 0.78]
   1    dog   → [0.19, -0.38, 0.82]
   2    fish  → [0.56,  0.12, -0.34]
   3    bird  → [0.45,  0.23, -0.12]
   4    plane → [-0.67, 0.89, 0.11]

  这张表就是一个 (5, 3) 的矩阵！

第 2 步：查表

  输入词 ID: 2 (fish)
  结果: [0.56, 0.12, -0.34]

  就这么简单：Embedding = 查表！
```

### Embedding 等价于 One-hot + 矩阵乘法

数学上：`Embedding(id)` = `one_hot(id) @ weight_matrix`。但查表只需读取一行，比矩阵乘法快上万倍。

### Embedding 层在神经网络中的位置

![Embedding层](images/embedding_layer.png)

*图5：Embedding 层在神经网络中的位置*

```
在大模型中，Embedding 是第一层：

  文本输入 → Tokenizer → Embedding 层 → Transformer 层
              (分词+编号)   (ID→向量)      (变换处理...)

  "I like cats"
       |
  Tokenizer：分词并分配 ID
       |
  [15, 234, 89]                  ← 3 个词的 ID
       |
  Embedding 查表
       |
  [[0.12, -0.34, ..., 0.56],    ← "I" 的向量     (256维)
   [0.78, 0.23, ..., -0.12],    ← "like" 的向量   (256维)
   [0.45, -0.67, ..., 0.89]]    ← "cats" 的向量   (256维)
       |
  shape: (3, 256)
```

**重要：Embedding 层的权重是可训练的！** 初始化时随机，训练中通过反向传播不断调整，训练后学到有意义的词向量。

**动手验证**：让我们用 PyTorch 代码体验 Embedding 查表。

In [ ]:
# =============================================
# 4.1 nn.Embedding 的基本用法
# =============================================

embed_dim = 4  # 嵌入维度（这里用小维度便于观察）
embedding = nn.Embedding(vocab_size, embed_dim)

print(f"嵌入层: Embedding({vocab_size}, {embed_dim})")
print(f"参数量: {vocab_size} x {embed_dim} = {vocab_size * embed_dim}")
print(f"\n嵌入矩阵 (权重):")
print(embedding.weight.data)

# 通过索引查找词向量
cat_idx = torch.tensor(word_to_idx["cat"])
cat_vec = embedding(cat_idx)
print(f"\n'cat' 的索引: {cat_idx.item()}")
print(f"'cat' 的嵌入向量: {cat_vec.data}")

# 批量查找
sentence_idx = torch.tensor([word_to_idx["cat"], word_to_idx["dog"], word_to_idx["fish"]])
sentence_vec = embedding(sentence_idx)
print(f"\n批量查找 ['cat', 'dog', 'fish']:")
print(f"  输入索引: {sentence_idx.tolist()}")
print(f"  输出形状: {sentence_vec.shape}  [序列长度=3, 嵌入维度={embed_dim}]")

In [ ]:
# =============================================
# 4.2 验证: Embedding 查表 = One-Hot × 矩阵乘法
# =============================================

# 方法1: 用 nn.Embedding 查找
idx = torch.tensor(word_to_idx["king"])
vec_embed = embedding(idx)

# 方法2: 用 One-Hot @ 矩阵乘法（等价但低效）
one_hot = torch.zeros(vocab_size)
one_hot[word_to_idx["king"]] = 1.0
vec_matmul = one_hot @ embedding.weight

print("两种方式结果相同:")
print(f"  Embedding 查找:    {vec_embed.data}")
print(f"  One-Hot 矩阵乘法: {vec_matmul.data}")
print(f"  是否相等: {torch.allclose(vec_embed, vec_matmul)}")

print(f"\n但 Embedding 更高效:")
print(f"  Embedding: 直接取第 {idx.item()} 行，O(1)")
print(f"  矩阵乘法: 需要 {vocab_size} 次乘加运算，O(V * d)")
print("  当词表有5万个词时，Embedding 比矩阵乘法快上万倍!")

In [ ]:
# =============================================
# 4.3 随机初始化的 Embedding 也没有语义
# =============================================

print("随机初始化的嵌入向量 - 余弦相似度:")
print("=" * 50)
for w1, w2 in pairs:
    v1 = embedding(torch.tensor(word_to_idx[w1]))
    v2 = embedding(torch.tensor(word_to_idx[w2]))
    sim = torch.nn.functional.cosine_similarity(v1.unsqueeze(0), v2.unsqueeze(0)).item()
    print(f"  {w1:>6} vs {w2:<6}  相似度 = {sim:.4f}")

print("\n现在相似度不全是 0 了（因为是随机的），但也没有语义意义")
print("接下来我们通过训练来让嵌入向量学到真正的语义!")

> 💡 **记忆要点**
> - Embedding 层本质上是一个**查表操作**：根据词 ID 查出对应的向量
> - 数学上等价于 **One-hot × 权重矩阵**，但查表更高效
> - Embedding 矩阵的 shape = **(vocab_size, embed_dim)**
> - 权重是**可训练的**，通过反向传播自动学习有意义的词向量

---
## 5. Word2Vec：从上下文中学习词义

Word2Vec 是 2013 年 Google 提出的经典词嵌入方法，有两种训练方式：

![Word2Vec训练](images/word2vec_training.png)

*图3：Word2Vec 的 CBOW 和 Skip-gram 训练方式*

**方式1：CBOW（Continuous Bag of Words）** — 用上下文预测中心词

```
训练样本: "I like eating apples and bananas"

  上下文词         →  预测中心词
  [I, eating]     →  like
  [like, apples]  →  eating

  类比：做完形填空！
  "I ____ eating apples" → 答案: "like"
```

**方式2：Skip-gram** — 用中心词预测上下文（和 CBOW 反过来）

```
  中心词       →  预测上下文
  like         →  [I, eating]
  apples       →  [eating, and]

  类比：给你 "apple"，猜猜它旁边会出现什么词？
```

**CBOW vs Skip-gram:**

| | CBOW | Skip-gram |
|---|---|---|
| 方向 | 上下文 → 中心词 | 中心词 → 上下文 |
| 类比 | 完形填空 | 自由联想 |
| 速度 | 更快，适合大数据 | 质量更好，尤其对低频词 |

**动手训练**：我们来用 Skip-gram 在一个小语料上训练词向量。

In [ ]:
# =============================================
# 5.1 准备训练语料
# =============================================

corpus = [
    "the king rules the kingdom",
    "the queen rules the kingdom",
    "the king wears a crown",
    "the queen wears a crown",
    "the man works hard",
    "the woman works hard",
    "the prince is the son of the king",
    "the princess is the daughter of the queen",
    "a cat is a pet",
    "a dog is a pet",
    "the cat chases a mouse",
    "the dog chases a cat",
    "a cat sits on the mat",
    "a dog sits on the rug",
    "the boy plays with the dog",
    "the girl plays with the cat",
    "the king and queen live in the castle",
    "the prince and princess live in the castle",
    "the man and woman walk in the park",
    "the boy and girl walk in the park",
]

# 分词
tokenized = [sentence.split() for sentence in corpus]

# 构建词表
word_counts = Counter(word for sentence in tokenized for word in sentence)
w2v_vocab = sorted(word_counts.keys())
w2v_word_to_idx = {word: idx for idx, word in enumerate(w2v_vocab)}
w2v_idx_to_word = {idx: word for word, idx in w2v_word_to_idx.items()}
w2v_vocab_size = len(w2v_vocab)

print(f"语料库: {len(corpus)} 个句子")
print(f"词表大小: {w2v_vocab_size}")
print(f"词频 Top-10: {word_counts.most_common(10)}")

In [ ]:
# =============================================
# 5.2 生成 Skip-gram 训练数据
# =============================================
# Skip-gram 的训练数据 = (中心词, 上下文词) 的配对
# window_size 决定了"上下文"的范围：中心词前后各 window_size 个词

def create_skipgram_data(tokenized_sentences, word_to_idx, window_size=2):
    """生成 Skip-gram 训练样本

    原理：对每个句子中的每个词（中心词），取它前后 window_size 个词作为上下文
    每个 (中心词, 上下文词) 配对就是一条训练样本

    例如句子 "the king rules the kingdom"，window_size=2
    当中心词是 "rules" 时：
      上下文词 = ["the", "king", "the", "kingdom"]
      生成 4 条样本: (rules, the), (rules, king), (rules, the), (rules, kingdom)
    """
    pairs = []
    for sentence in tokenized_sentences:
        # 把句子中的每个词转成索引号
        indices = [word_to_idx[w] for w in sentence]

        # 遍历句子中的每个位置作为中心词
        for center_pos in range(len(indices)):
            # 遍历窗口内的每个偏移量（-2, -1, 1, 2）
            for offset in range(-window_size, window_size + 1):
                context_pos = center_pos + offset

                # 跳过自身（offset=0），跳过越界的位置
                if offset == 0 or context_pos < 0 or context_pos >= len(indices):
                    continue

                # 记录 (中心词索引, 上下文词索引) 配对
                pairs.append((indices[center_pos], indices[context_pos]))
    return pairs

window_size = 2
train_pairs = create_skipgram_data(tokenized, w2v_word_to_idx, window_size)

print(f"窗口大小: {window_size}")
print(f"训练样本数: {len(train_pairs)}")
print(f"\n前10个训练样本 (中心词 → 上下文词):")
for center, context in train_pairs[:10]:
    print(f"  {w2v_idx_to_word[center]:>10} → {w2v_idx_to_word[context]}")

# 转成张量，方便后续训练
center_words = torch.tensor([p[0] for p in train_pairs])
context_words = torch.tensor([p[1] for p in train_pairs])

In [ ]:
# =============================================
# 5.3 定义 Skip-gram 模型
# =============================================

class SkipGram(nn.Module):
    """Skip-gram Word2Vec 模型

    思路：
      输入一个中心词的索引 → 查嵌入表得到词向量 → 用线性层预测上下文词

    结构：
      center_embedding: 嵌入层，把词索引映射成向量（这就是我们要学的词向量！）
      output_layer:     线性层，把词向量映射回词表大小，产生每个词的"得分"
                        得分最高的词就是模型预测的上下文词

    训练完成后，center_embedding 的权重就是学到的词向量
    """
    def __init__(self, vocab_size, embed_dim):
        super().__init__()
        # 嵌入层：vocab_size 个词，每个词用 embed_dim 维的向量表示
        self.center_embedding = nn.Embedding(vocab_size, embed_dim)
        # 输出层：把 embed_dim 维向量映射成 vocab_size 维的得分
        # 得分经过 CrossEntropyLoss 会自动做 softmax，变成概率分布
        self.output_layer = nn.Linear(embed_dim, vocab_size)

    def forward(self, center_word_idx):
        # 第1步：查表，把词索引变成词向量
        embed = self.center_embedding(center_word_idx)  # [batch] → [batch, embed_dim]
        # 第2步：线性变换，预测每个词是上下文词的得分
        scores = self.output_layer(embed)  # [batch, embed_dim] → [batch, vocab_size]
        return scores

# 嵌入维度：每个词用 32 个数字表示（实际应用中通常用 100-300）
embed_dim = 32
model = SkipGram(w2v_vocab_size, embed_dim)

print(f"模型结构:")
print(model)
total_params = sum(p.numel() for p in model.parameters())
print(f"\n总参数量: {total_params}")
print(f"  嵌入层: {w2v_vocab_size} × {embed_dim} = {w2v_vocab_size * embed_dim}")
print(f"  输出层: {embed_dim} × {w2v_vocab_size} + {w2v_vocab_size} = {embed_dim * w2v_vocab_size + w2v_vocab_size}")

In [ ]:
# =============================================
# 5.4 训练 Skip-gram 模型
# =============================================
# 训练目标：给定中心词，模型要能预测出正确的上下文词
# 损失函数：Cross-Entropy Loss（和分类任务一样，把"预测上下文词"看作多分类问题）
# 优化器：Adam（一种自适应学习率的优化器，比 SGD 更稳定、更快收敛）

learning_rate = 0.01
n_epochs = 200       # 训练 200 轮
batch_size = 64      # 每次取 64 个样本计算一次梯度

# CrossEntropyLoss = softmax + 交叉熵，是多分类问题的标准损失函数
criterion = nn.CrossEntropyLoss()
# Adam 优化器：自动为每个参数调节学习率，实际项目中最常用的优化器
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

losses = []
n_samples = len(center_words)

print(f"开始训练 Skip-gram...")
print(f"训练样本: {n_samples}, 批大小: {batch_size}, 轮数: {n_epochs}")
print(f"{'Epoch':>6} {'Loss':>10}")
print("-" * 20)

for epoch in range(n_epochs):
    # 每轮开始时打乱数据顺序，避免模型记住数据的排列规律
    perm = torch.randperm(n_samples)
    epoch_loss = 0.0
    n_batches = 0

    # 按批次遍历所有训练数据
    for i in range(0, n_samples, batch_size):
        # 取出当前批次的数据索引
        batch_idx = perm[i:i+batch_size]
        batch_center = center_words[batch_idx]    # 这一批的中心词
        batch_context = context_words[batch_idx]  # 这一批的上下文词（正确答案）

        # 前向传播：输入中心词，得到每个词的预测得分
        scores = model(batch_center)  # [batch_size, vocab_size]

        # 计算损失：预测得分 vs 正确的上下文词
        # CrossEntropyLoss 会自动做 softmax，不需要手动转概率
        loss = criterion(scores, batch_context)

        # 反向传播 + 参数更新（第3章学过的五步法）
        optimizer.zero_grad()  # 清空上一步的梯度
        loss.backward()        # 计算梯度
        optimizer.step()       # 更新参数

        epoch_loss += loss.item()
        n_batches += 1

    avg_loss = epoch_loss / n_batches
    losses.append(avg_loss)

    if (epoch + 1) % 40 == 0:
        print(f"{epoch+1:>6} {avg_loss:>10.4f}")

print(f"\n训练完成! 最终损失: {losses[-1]:.4f}")

In [ ]:
# =============================================
# 5.5 可视化训练损失
# =============================================

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(losses, color='steelblue', linewidth=2)
ax.set_title('Skip-gram Training Loss', fontsize=14)
ax.set_xlabel('Epoch')
ax.set_ylabel('Cross-Entropy Loss')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("损失逐渐下降，说明模型在学习词与词之间的共现关系")

> 💡 **记忆要点**
> - Word2Vec 有两种方式：**CBOW**（上下文→中心词）和 **Skip-gram**（中心词→上下文）
> - 核心思路：通过预测上下文关系来学习词义
> - 训练过程和上一章学的完全一样：前向→损失→反向→更新
> - 训练完成后，语义相近的词向量会自动变得接近

### 探索自训练的词向量

模型训练完成后，嵌入层中的权重就是学到的词向量。让我们看看这些向量是否捕捉到了语义关系。

In [ ]:
# =============================================
# 5.6 提取词向量并计算相似度
# =============================================
# 训练完成后，嵌入层的权重矩阵就是学到的词向量
# .data.clone() 是为了复制一份，不影响原始模型参数

word_vectors = model.center_embedding.weight.data.clone()
print(f"词向量矩阵形状: {word_vectors.shape}  [{w2v_vocab_size} 个词, {embed_dim} 维]")

def find_similar(word, word_vectors, word_to_idx, idx_to_word, top_k=5):
    """查找与给定词最相似的 top_k 个词

    原理：计算目标词向量与所有词向量的余弦相似度，取最大的几个

    余弦相似度 = 两个向量夹角的余弦值
      1.0 = 方向完全相同（语义非常接近）
      0.0 = 正交（不相关）
     -1.0 = 方向相反（语义相反）
    """
    if word not in word_to_idx:
        print(f"'{word}' 不在词表中")
        return

    # 取出目标词的向量，unsqueeze(0) 添加 batch 维度以便计算
    word_vec = word_vectors[word_to_idx[word]].unsqueeze(0)  # [1, embed_dim]

    # 计算与所有词的余弦相似度
    # cosine_similarity 自动处理广播：[1, dim] vs [vocab_size, dim] → [vocab_size]
    sims = torch.nn.functional.cosine_similarity(word_vec, word_vectors)

    # 取相似度最高的 top_k+1 个（+1 是因为最相似的是自己）
    values, indices = torch.topk(sims, top_k + 1)

    print(f"与 '{word}' 最相似的词:")
    for val, idx in zip(values, indices):
        w = idx_to_word[idx.item()]
        if w != word:  # 排除自己
            print(f"  {w:<12} 相似度: {val.item():.4f}")

# 测试几个词
for word in ["king", "cat", "man"]:
    find_similar(word, word_vectors, w2v_word_to_idx, w2v_idx_to_word)
    print()

In [ ]:
# =============================================
# 5.7 可视化自训练的词向量
# =============================================
# 用 PCA 把 32 维的词向量降到 2 维，画在平面上
# 如果训练有效，语义相近的词应该聚集在一起

pca = PCA(n_components=2)
vectors_2d = pca.fit_transform(word_vectors.numpy())

fig, ax = plt.subplots(figsize=(12, 8))

# 按类别给词标不同颜色，便于观察聚类效果
categories = {
    'royalty': ['king', 'queen', 'prince', 'princess', 'crown', 'castle', 'kingdom'],
    'people': ['man', 'woman', 'boy', 'girl'],
    'animals': ['cat', 'dog', 'mouse', 'pet'],
    'other': [],
}
# 把不在上面任何类别中的词归入 "other"
categorized = set()
for words in categories.values():
    categorized.update(words)
categories['other'] = [w for w in w2v_vocab if w not in categorized]

colors_map = {'royalty': 'crimson', 'people': 'forestgreen', 'animals': 'dodgerblue', 'other': 'gray'}

# 画每个词的点和标注
for cat, words in categories.items():
    for word in words:
        if word in w2v_word_to_idx:
            idx = w2v_word_to_idx[word]
            ax.scatter(vectors_2d[idx, 0], vectors_2d[idx, 1], c=colors_map[cat], s=80, zorder=5)
            ax.annotate(word, (vectors_2d[idx, 0], vectors_2d[idx, 1]),
                       textcoords="offset points", xytext=(6, 6), fontsize=10)

# 添加图例（用空散点只是为了生成图例标签）
for cat, color in colors_map.items():
    if cat != 'other':
        ax.scatter([], [], c=color, s=80, label=cat)
ax.scatter([], [], c='gray', s=80, label='other')

ax.set_title('Self-Trained Word2Vec Embeddings (PCA)', fontsize=14)
ax.legend(loc='best', fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("观察: 即使语料很小，语义相关的词已经开始聚集在一起")

---
## 6. 词向量的几何魔法：国王 - 男人 + 女人 ≈ 女王

我们自训练的模型语料太小，无法可靠地展示类比关系。现在加载在**大规模语料**上训练好的 **GloVe 预训练词向量**来验证这个经典等式。

### 最惊人的发现

```
词向量可以做加减法，而且结果在语义上是合理的！

  vector("King") - vector("Man") + vector("Woman") ≈ vector("Queen")
```

### 几何解释

![词向量类比](images/word_analogy.png)

*图4："国王 - 男人 + 女人 = 女王"的向量空间可视化*

```
在词向量空间中，性别关系形成平行方向：

  维度 2
  (皇室)
  |
  |  . Queen  - - - - . King
  |       ^                ^
  |       | 性别           | 性别
  |       |                |
  |  . Woman  - - - - . Man
  |
  +------------------------------> 维度 1（性别）

  关键发现：
  King - Man = 一个"皇室"方向向量
  Woman + "皇室方向" = Queen！
```

### 更多类比关系

| 关系类型 | 等式 |
|----------|------|
| 性别 | King - Man + Woman ≈ Queen |
| 国家-首都 | Paris - France + Italy ≈ Rome |
| 时态 | walking - walk + swim ≈ swimming |
| 比较级 | bigger - big + small ≈ smaller |
| 公司-产品 | Apple - iPhone + Windows ≈ Microsoft |

### 衡量词语相似度

```
余弦相似度：

              A · B
  cos(θ) = ---------
            |A| × |B|

  取值范围: [-1, 1]
   1  = 方向相同（非常相似）
   0  = 正交（不相关）
  -1  = 方向相反（含义相反）
```

**动手验证**：让我们用 GloVe 预训练向量来亲手体验这些魔法。

In [ ]:
# =============================================
# 6.1 加载预训练 GloVe 向量
# =============================================
# GloVe (Global Vectors) 是斯坦福大学在大规模语料上训练的词向量
# 我们加载 50 维版本（glove-wiki-gigaword-50），包含 40 万个词
#
# 加载策略：优先用 gensim（简洁），失败则自动切换到手动下载（兜底）

import sys

# --- 方法一：尝试用 gensim 加载 ---
try:
    # 清除可能已缓存的旧版本模块，确保使用 cell-2 新安装的版本
    for mod in list(sys.modules):
        if 'smart_open' in mod or 'gensim' in mod:
            del sys.modules[mod]

    import gensim.downloader as api

    print("正在通过 gensim 加载 GloVe 词向量 (glove-wiki-gigaword-50)...")
    print("首次运行需要下载约 66MB，请耐心等待...")
    glove = api.load("glove-wiki-gigaword-50")

    print(f"\n加载完成!")
    print(f"词表大小: {len(glove)} 个词")
    print(f"向量维度: {glove.vector_size}")

except Exception as e:
    # --- 方法二：手动下载 GloVe（兜底方案） ---
    print(f"gensim 不可用 ({type(e).__name__}: {e})")
    print("切换到手动下载方式...\n")

    import urllib.request
    import zipfile
    import os

    class GloVeVectors:
        """GloVe 词向量封装类

        提供与 gensim KeyedVectors 相同的常用 API：
          glove[word]                    → 获取词向量
          glove.similarity(w1, w2)       → 计算两个词的余弦相似度
          glove.most_similar(word)       → 查找最相似的词
          glove.most_similar(positive=, negative=)  → 类比推理
        """

        def __init__(self, vectors_dict, dim):
            self.vectors = vectors_dict
            self.vector_size = dim

            # 将所有词向量堆叠成矩阵，便于批量计算相似度
            self.words = list(vectors_dict.keys())
            self.matrix = np.stack([vectors_dict[w] for w in self.words])

            # 预计算归一化向量（除以 L2 范数），加速余弦相似度计算
            # 归一化后，向量点积 = 余弦相似度
            norms = np.linalg.norm(self.matrix, axis=1, keepdims=True)
            self.normed = self.matrix / norms

            # 词到索引的映射，用于快速查找
            self._word_to_idx = {w: i for i, w in enumerate(self.words)}

        def __getitem__(self, word):
            return self.vectors[word]

        def __contains__(self, word):
            return word in self.vectors

        def __len__(self):
            return len(self.vectors)

        def similarity(self, w1, w2):
            """计算两个词的余弦相似度"""
            v1, v2 = self.vectors[w1], self.vectors[w2]
            return float(np.dot(v1, v2) / (np.linalg.norm(v1) * np.linalg.norm(v2)))

        def most_similar(self, positive=None, negative=None, topn=10):
            """查找最相似的词，支持类比推理

            用法1: most_similar("king", topn=5)         → 与 king 最相似的 5 个词
            用法2: most_similar(positive=["king", "woman"], negative=["man"])
                   → king - man + woman ≈ ? （类比推理）
            """
            # 统一处理：允许传入单个字符串
            if isinstance(positive, str):
                positive = [positive]
            positive = positive or []
            negative = negative or []

            # 构造目标向量：positive 词向量之和 - negative 词向量之和
            target = np.zeros(self.vector_size, dtype=np.float32)
            for w in positive:
                target += self.vectors[w]
            for w in negative:
                target -= self.vectors[w]

            # 归一化目标向量，然后与所有词的归一化向量做点积 = 余弦相似度
            target_normed = target / np.linalg.norm(target)
            sims = self.normed @ target_normed  # shape: [vocab_size]

            # 按相似度降序排列，排除输入词本身
            exclude = set(positive + negative)
            results = []
            for idx in np.argsort(sims)[::-1]:
                w = self.words[idx]
                if w not in exclude:
                    results.append((w, float(sims[idx])))
                    if len(results) >= topn:
                        break
            return results

    # --- 下载 GloVe 文件 ---
    glove_path = '/tmp/glove.6B.50d.txt'

    if not os.path.exists(glove_path):
        zip_path = '/tmp/glove.6B.zip'
        url = 'https://nlp.stanford.edu/data/glove.6B.zip'
        print("正在从 Stanford NLP 下载 GloVe 词向量 (~822MB)...")
        print("首次运行需要几分钟，后续运行会使用缓存。\n")
        urllib.request.urlretrieve(url, zip_path)
        print("下载完成! 正在解压 50 维版本...")
        with zipfile.ZipFile(zip_path) as zf:
            zf.extract('glove.6B.50d.txt', '/tmp/')
        os.remove(zip_path)  # 删除 zip 节省空间

    # --- 加载词向量到内存 ---
    print("正在解析词向量文件...")
    vectors = {}
    with open(glove_path, encoding='utf-8') as f:
        for line in f:
            parts = line.strip().split()
            word = parts[0]
            # 每行格式: "word 0.123 -0.456 ..." → 解析为 numpy 数组
            vectors[word] = np.array(parts[1:], dtype=np.float32)

    glove = GloVeVectors(vectors, len(next(iter(vectors.values()))))

    print(f"\n加载完成!")
    print(f"词表大小: {len(glove)} 个词")
    print(f"向量维度: {glove.vector_size}")

king_vec = glove['king']
print(f"\n'king' 的向量 (前10维): {king_vec[:10]}")

In [ ]:
# =============================================
# 6.2 计算词语相似度
# =============================================

print("词对余弦相似度:")
print("=" * 50)

word_pairs = [
    ("king", "queen"), ("cat", "dog"), ("good", "great"), ("man", "woman"),
    ("king", "banana"), ("cat", "economics"), ("happy", "table"),
]

for w1, w2 in word_pairs:
    sim = glove.similarity(w1, w2)
    bar = '|' * int(abs(sim) * 30)
    print(f"  {w1:>10} vs {w2:<12} {sim:>7.4f}  {bar}")

print("\n语义相近的词对相似度高，语义无关的词对相似度低")

In [ ]:
# =============================================
# 6.3 查找最相似的词
# =============================================

for word in ["king", "computer", "happy", "dog"]:
    similar = glove.most_similar(word, topn=5)
    print(f"与 '{word}' 最相似的5个词:")
    for w, score in similar:
        print(f"    {w:<15} {score:.4f}")
    print()

In [ ]:
# =============================================
# 6.4 词向量类比: 国王 - 男人 + 女人 ≈ 女王
# =============================================
# most_similar 函数做的事情：
#   1. 计算 positive 词向量之和 - negative 词向量 = 目标向量
#   2. 在所有词中找与目标向量最接近的词
# 例如: positive=["king", "woman"], negative=["man"]
#   → 计算 vector("king") + vector("woman") - vector("man")
#   → 找到最接近的词 → "queen"

print("词向量类比推理:")
print("=" * 60)

analogies = [
    ("king", "man", "woman",    "king - man + woman ≈ ?"),
    ("queen", "woman", "man",   "queen - woman + man ≈ ?"),
    ("paris", "france", "italy", "paris - france + italy ≈ ?"),
    ("bigger", "big", "small",  "bigger - big + small ≈ ?"),
    ("walking", "walk", "run",  "walking - walk + run ≈ ?"),
]

for pos1, neg, pos2, desc in analogies:
    try:
        # positive: 要加的词, negative: 要减的词
        # topn=3: 返回最相似的前3个词
        result = glove.most_similar(positive=[pos1, pos2], negative=[neg], topn=3)
        answers = ", ".join([f"{w} ({s:.3f})" for w, s in result])
        print(f"\n  {desc}")
        print(f"    答案: {answers}")
    except KeyError as e:
        print(f"  {desc} → 词不在词表中: {e}")

print("\n词向量的类比能力说明它学到了有意义的语义方向:")
print("  - '性别'方向: man→woman, king→queen")
print("  - '国家-首都'方向: france→paris, italy→rome")

In [ ]:
# =============================================
# 6.5 手动验证类比运算
# =============================================
# 上面用了 gensim 的封装函数，现在我们手动用向量运算来验证
# 核心等式: King - Man + Woman ≈ Queen

# 取出四个词的向量
v_king = torch.tensor(glove['king'])
v_man = torch.tensor(glove['man'])
v_woman = torch.tensor(glove['woman'])
v_queen = torch.tensor(glove['queen'])

# 手动计算: king - man + woman
v_result = v_king - v_man + v_woman

# 计算结果向量与 queen 的余弦相似度
# unsqueeze(0) 是因为 cosine_similarity 要求至少 2 维输入
sim_queen = torch.nn.functional.cosine_similarity(
    v_result.unsqueeze(0), v_queen.unsqueeze(0)
).item()

print("手动向量运算验证:")
print(f"  king - man + woman 与 queen 的相似度: {sim_queen:.4f}")

# 对比结果向量与多个词的相似度，看 queen 是否排名最高
compare_words = ['queen', 'king', 'princess', 'prince', 'woman', 'man', 'table']
print(f"\n  king - man + woman 与各词的相似度:")
for word in compare_words:
    v_word = torch.tensor(glove[word])
    sim = torch.nn.functional.cosine_similarity(
        v_result.unsqueeze(0), v_word.unsqueeze(0)
    ).item()
    # 画一个简单的条形图，直观显示相似度高低
    bar = '|' * int(sim * 30) if sim > 0 else ''
    marker = " ★" if word == 'queen' else ""
    print(f"    {word:<12} {sim:.4f}  {bar}{marker}")

print("\nqueen 的相似度最高，验证了 king - man + woman ≈ queen!")

> 💡 **记忆要点**
> - 词向量可以做加减法：**国王 - 男人 + 女人 ≈ 女王**
> - 这反映了词向量空间中的**规律性几何结构**
> - 不同的语义关系（性别、国家-首都、时态等）对应空间中不同的方向
> - 用**余弦相似度**衡量词语的语义相似性，值域 [-1, 1]

### 可视化预训练词向量

让我们把 GloVe 词向量降维到 2D，直观地看看语义关系。

In [ ]:
# =============================================
# 6.6 可视化 GloVe 词向量的语义空间
# =============================================
# 选取 5 组不同类别的词，用 PCA 降到 2 维后画在平面上
# 预期效果：同类别的词应该聚集在一起

# 定义 5 组词，每组 6 个
word_groups = {
    'royalty': ['king', 'queen', 'prince', 'princess', 'throne', 'crown'],
    'gender': ['man', 'woman', 'boy', 'girl', 'father', 'mother'],
    'animals': ['cat', 'dog', 'fish', 'bird', 'horse', 'lion'],
    'countries': ['china', 'japan', 'france', 'germany', 'brazil', 'india'],
    'numbers': ['one', 'two', 'three', 'four', 'five', 'six'],
}

# 收集所有词的向量
all_words = []
all_vectors = []
word_to_group = {}

for group, words in word_groups.items():
    for word in words:
        if word in glove:
            all_words.append(word)
            all_vectors.append(glove[word])      # 取出预训练的 50 维词向量
            word_to_group[word] = group

# PCA 降维: 50 维 → 2 维
vectors_matrix = np.array(all_vectors)
pca = PCA(n_components=2)
vectors_2d = pca.fit_transform(vectors_matrix)

fig, ax = plt.subplots(figsize=(14, 10))

# 每组用不同颜色
group_colors = {
    'royalty': 'crimson', 'gender': 'forestgreen',
    'animals': 'dodgerblue', 'countries': 'darkorange', 'numbers': 'purple',
}

# 画每个词的点和标注
for i, word in enumerate(all_words):
    group = word_to_group[word]
    color = group_colors[group]
    ax.scatter(vectors_2d[i, 0], vectors_2d[i, 1], c=color, s=100, zorder=5)
    ax.annotate(word, (vectors_2d[i, 0], vectors_2d[i, 1]),
               textcoords="offset points", xytext=(6, 6), fontsize=10)

# 添加图例
for group, color in group_colors.items():
    ax.scatter([], [], c=color, s=100, label=group)

ax.set_title('GloVe Word Embeddings 2D Visualization (PCA)', fontsize=16)
ax.legend(loc='best', fontsize=12)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("观察: 同一类别的词聚集在一起，词向量成功编码了语义信息")

In [ ]:
# =============================================
# 6.7 可视化类比关系的向量运算
# =============================================
# 画出"性别方向"：从男性词到女性词的箭头
# 如果这些箭头大致平行，说明词向量空间中存在统一的"性别"方向
# 这就是 king - man + woman ≈ queen 成立的几何原因

# 选取 5 对性别相关的词
analogy_words = ['king', 'queen', 'man', 'woman', 'prince', 'princess',
                 'father', 'mother', 'boy', 'girl']

# 取出它们的 GloVe 向量，PCA 降到 2 维
analogy_vectors = np.array([glove[w] for w in analogy_words])
pca_analogy = PCA(n_components=2)
analogy_2d = pca_analogy.fit_transform(analogy_vectors)

fig, ax = plt.subplots(figsize=(12, 8))

# 画每个词的点，皇室词用红色，普通词用绿色
for i, word in enumerate(analogy_words):
    color = 'crimson' if word in ['king', 'queen', 'prince', 'princess'] else 'forestgreen'
    ax.scatter(analogy_2d[i, 0], analogy_2d[i, 1], c=color, s=150, zorder=5)
    ax.annotate(word, (analogy_2d[i, 0], analogy_2d[i, 1]),
               textcoords="offset points", xytext=(8, 8), fontsize=13, fontweight='bold')

# 画从男性词指向女性词的箭头
gender_pairs = [('king', 'queen'), ('man', 'woman'), ('prince', 'princess'),
                ('father', 'mother'), ('boy', 'girl')]

for m, f in gender_pairs:
    m_idx = analogy_words.index(m)
    f_idx = analogy_words.index(f)
    # annotate 的箭头：从 xytext(男) 指向 xy(女)
    ax.annotate('', xy=analogy_2d[f_idx], xytext=analogy_2d[m_idx],
               arrowprops=dict(arrowstyle='->', color='gray', lw=1.5, alpha=0.6))

ax.set_title('Gender Direction in Word Vector Space', fontsize=14)
ax.grid(True, alpha=0.3)
# 在图的左上角添加说明文字
ax.text(0.02, 0.98, '箭头表示男→女的方向\n（近似平行！）',
        transform=ax.transAxes, fontsize=11, verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
plt.tight_layout()
plt.show()

print("关键发现: 所有男→女的箭头方向大致平行!")
print("  这就是为什么 king - man + woman ≈ queen 成立!")

---
## 7. Embedding 在大模型中的角色

在 GPT/BERT 等大模型中，输入嵌入 = **词嵌入 (Token Embedding)** + **位置嵌入 (Position Embedding)**。

```
Text Input → Tokenizer → Embedding Layer → Transformer Layers
              (split+ID)    (ID→vector)       (transform...)
```

让我们模拟这个过程，并对比不同模型的嵌入参数量。

In [ ]:
# =============================================
# 7.1 模拟 Transformer 的输入嵌入
# =============================================

vocab_size_sim = 10000
max_seq_len = 512
d_model = 768  # GPT-2 small 用的就是 768

token_embedding = nn.Embedding(vocab_size_sim, d_model)
position_embedding = nn.Embedding(max_seq_len, d_model)

# 模拟输入
input_ids = torch.tensor([[101, 2003, 1037, 6251, 102]])  # [batch=1, seq_len=5]
seq_len = input_ids.shape[1]
position_ids = torch.arange(seq_len).unsqueeze(0)

# 计算嵌入
token_embeds = token_embedding(input_ids)
pos_embeds = position_embedding(position_ids)
final_embeds = token_embeds + pos_embeds

print("Transformer 输入嵌入过程:")
print(f"  输入词索引:   {input_ids.shape}  → [batch=1, seq_len=5]")
print(f"  词嵌入:       {token_embeds.shape}  → [batch, seq_len, d_model]")
print(f"  位置嵌入:     {pos_embeds.shape}  → [batch, seq_len, d_model]")
print(f"  最终嵌入:     {final_embeds.shape}  → 送入 Transformer 层")

token_params = vocab_size_sim * d_model
pos_params = max_seq_len * d_model
total = token_params + pos_params
print(f"\n嵌入层参数量:")
print(f"  词嵌入:   {vocab_size_sim} x {d_model} = {token_params:,}")
print(f"  位置嵌入: {max_seq_len} x {d_model} = {pos_params:,}")
print(f"  总计:     {total:,} ({total/1e6:.1f}M)")

In [ ]:
# =============================================
# 7.2 GPT 系列模型的嵌入参数对比
# =============================================

models = {
    'GPT-2 Small':  {'vocab': 50257, 'dim': 768,  'layers': 12, 'ctx': 1024},
    'GPT-2 Medium': {'vocab': 50257, 'dim': 1024, 'layers': 24, 'ctx': 1024},
    'GPT-2 Large':  {'vocab': 50257, 'dim': 1280, 'layers': 36, 'ctx': 1024},
    'GPT-3 (175B)': {'vocab': 50257, 'dim': 12288,'layers': 96, 'ctx': 2048},
}

print(f"{'Model':<16} {'Vocab':>8} {'Dim':>6} {'Embed Params':>14} {'Pct of Total':>14}")
print("=" * 62)

for name, cfg in models.items():
    embed_params = cfg['vocab'] * cfg['dim'] + cfg['ctx'] * cfg['dim']
    layer_params = 12 * cfg['dim'] ** 2
    total_est = embed_params + cfg['layers'] * layer_params
    pct = embed_params / total_est * 100
    print(f"{name:<16} {cfg['vocab']:>8,} {cfg['dim']:>6} {embed_params:>14,} {pct:>12.1f}%")

print("\n观察: 随着模型变大，嵌入层占总参数的比例逐渐降低")
print("  大模型的参数主要在 Transformer 层 (注意力 + 前馈网络)")

---
## 8. 本章总结

**1. 文字 → 数字** — 神经网络只能处理数字，好的编码让语义相近的词在数字上也相近

**2. One-hot 编码** — 一个位置是 1，其余全是 0。问题：维度爆炸、无法表达相似性、无法泛化

**3. 词嵌入 (Word Embedding)** — 用稠密、低维的向量表示每个词。核心假设：相似上下文 = 相似含义

**4. Word2Vec** — CBOW（上下文→中心词）和 Skip-gram（中心词→上下文），通过大量文本训练自动学习词义

**5. 词向量的几何魔法** — King - Man + Woman ≈ Queen，不同语义关系 = 空间中不同方向，余弦相似度衡量语义距离

**6. PyTorch Embedding 层** — 本质是查表（ID→向量），等价于 One-hot × 权重矩阵，shape = (vocab_size, embed_dim)，权重可训练，是大模型的第一层

### 下一章预告

在第5章中，我们将学习**注意力机制（Attention）**：
- 注意力机制如何让模型"关注"重要信息
- Self-Attention 的数学原理
- Multi-Head Attention 的直觉
- 从零实现注意力机制

---

> **参考资料**
> - Mikolov et al., 2013: *Efficient Estimation of Word Representations in Vector Space* — Word2Vec 原始论文
> - Pennington et al., 2014: *GloVe: Global Vectors for Word Representation* — 另一种经典词嵌入方法
> - Jay Alammar: *The Illustrated Word2Vec* — Word2Vec 可视化讲解
> - PyTorch Official Documentation: `nn.Embedding` — Embedding 层 API 文档